In [144]:
import kagglehub
import tensorflow as tf
import numpy as np
import cv2
import os
from sklearn.model_selection import train_test_split

In [145]:
base_dir='/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'

train_dir = os.path.join(base_dir, 'Training')
validation_dir = os.path.join(base_dir, 'Testing')

train_glioma_dir = os.path.join(train_dir, 'glioma')
train_meningioma_dir = os.path.join(train_dir, 'meningioma')
train_pituitary_dir = os.path.join(train_dir, 'pituitary')
train_notumor_dir = os.path.join(train_dir, 'notumor')

validation_glioma_dir = os.path.join(validation_dir, 'glioma')
validation_meningioma_dir = os.path.join(validation_dir, 'meningioma')
validation_pituitary_dir = os.path.join(validation_dir, 'pituitary')
validation_notumor_dir = os.path.join(validation_dir, 'notumor')

In [146]:
image_size = (128,128)
Batch_size = 32

In [147]:
def load_images_and_labels(directory , label):
    images = []
    labels = []

    image_files = [f for f in os.listdir(directory) if f.lower().endswith(('.png' , '.jpg' , '.jpeg'))]
    for filename in image_files:
        image_path = os.path.join(directory , filename)

        image = cv2.imread(image_path)
        image = cv2.resize(image, image_size)

        images.append(image)
        labels.append(label)
    return images , labels

In [148]:
x_train_glioma , y_train_glioma = load_images_and_labels( train_glioma_dir , 0 )
x_train_meningioma , y_train_meningioma = load_images_and_labels( train_meningioma_dir , 1 )
x_train_pituitary , y_train_pituitary = load_images_and_labels( train_pituitary_dir , 2 )
x_train_notumor , y_train_notumor = load_images_and_labels( train_notumor_dir , 3 )

In [149]:
x_validation_glioma , y_validation_glioma = load_images_and_labels( validation_glioma_dir , 0 )
x_validation_meningioma , y_validation_meningioma = load_images_and_labels( validation_meningioma_dir , 1 )
x_validation_pituitary , y_validation_pituitary = load_images_and_labels( validation_pituitary_dir , 2 )
x_validation_notumor , y_validation_notumor = load_images_and_labels( validation_notumor_dir , 3 )

In [150]:
x_train = np.array(x_train_glioma + x_train_meningioma + x_train_pituitary + x_train_notumor)
y_train = np.array(y_train_glioma + y_train_meningioma + y_train_pituitary + y_train_notumor)

x_validation = np.array(x_validation_glioma + x_validation_meningioma + x_validation_pituitary + x_validation_notumor)
y_validation = np.array(y_validation_glioma + y_validation_meningioma + y_validation_pituitary + y_validation_notumor)

In [151]:
def shuffle(images , labels):
    combined = list(zip(images , labels))
    np.random.shuffle(combined)
    shuffled_images , shuffled_labels = zip(*combined)
    return np.array(shuffled_images) , np.array(shuffled_labels)

In [152]:
x_train , y_train = shuffle(x_train , y_train)
x_validation , y_validation = shuffle(x_validation , y_validation)

In [153]:
x_train = x_train.astype('float32') / 255.0
x_validation = x_validation.astype('float32') / 255.0

In [154]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(128, 128, 3)),


    
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),


    tf.keras.layers.Dense(4, activation='softmax')
])

model.summary()

Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_23 (Flatten)            │ (None, 49152)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_97 (Dense)                │ (None, 512)            │    25,166,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_74 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_98 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_75 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_99 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_76 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_100 (Dense)               │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,331,076 (96.63 MB)

 Trainable params: 25,331,076 (96.63 MB)

 Non-trainable params: 0 (0.00 B)

In [155]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=['accuracy'])

if len(x_train) > 0 and len(y_train) > 0 :
  history = model.fit(x_train , y_train ,
                      epochs=18,
                      batch_size=Batch_size,
                      validation_data=(x_validation, y_validation))


Epoch 1/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.4207 - loss: 2.1308 - val_accuracy: 0.5606 - val_loss: 1.0220
Epoch 2/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5152 - loss: 1.1013 - val_accuracy: 0.6363 - val_loss: 1.0198
Epoch 3/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5377 - loss: 1.0431 - val_accuracy: 0.6525 - val_loss: 0.9015
Epoch 4/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5891 - loss: 0.9409 - val_accuracy: 0.6275 - val_loss: 0.9651
Epoch 5/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5930 - loss: 0.9185 - val_accuracy: 0.6231 - val_loss: 0.9538
Epoch 6/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6114 - loss: 0.8969 - val_accuracy: 0.6587 - val_loss: 0.8784
Epoch 7/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6477 - loss: 0.8301 - val_accuracy: 0.6394 - val_loss: 0.9240
Epoch 8/18
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6637 - loss: 0.8058 - val_accu

In [157]:
model.save("model.keras")